# justounetsimple_opt_readonly: the DDR reads alone

`justounetsimple_opt` with the compute taken out. The four DDR readers are the same
(the same ports, bursts and rows), but instead of the patch buffer and the four conv
blocks, one process takes a word from every reader each cycle and adds it up. Nothing
waits for compute, so the time below is how fast the cube comes out of DDR.

Run the `justounetsimple_opt` notebook on the same image and compare its
**FPGA alone** line with this notebook's: if they are about the same, the reads are
the bottleneck; if this one is clearly faster, the compute is.

Next to this notebook: `justounetsimple_opt_readonly.bit` + `.hwh`, its register map
`x..._hw.h`, and the same image zip as for `justounetsimple_opt` (`aegean_unet.zip`).
Run the cells from the top.

## 1. Load the bitstream

In [ ]:
import re, time, zipfile
from pathlib import Path
import numpy as np
from pynq import Overlay, allocate, Clocks

# Which kernel: its register map, x<NAME>_hw.h, lies next to its NAME.bit.
header_file = next(Path('.').glob('x*_hw.h'))
NAME = header_file.name[1:-len('_hw.h')]
overlay = Overlay(f'{NAME}.bit')                 # also sets the FPGA clock from the .hwh
kernel = overlay.kernel                         # the HLS block, named 'kernel' in the block design

header = header_file.read_text()
def register(name):
    return int(re.search(rf'_ADDR_{name}\s+(0x[0-9a-fA-F]+)', header).group(1), 16)
AP_CTRL, HEIGHT, WIDTH = register('AP_CTRL'), register('HEIGHT_DATA'), register('WIDTH_DATA')
DINS = [int(a, 16) for a in re.findall(r'_ADDR_DIN\d+_DATA\s+(0x[0-9a-fA-F]+)', header)]   # din0, din1, ...
DOUT = register('DOUT_DATA')

# The formats (justounetsimple_opt_readonly.hpp): in, the raw cube (H x W x 120 uint16);
# out, one 128-bit word: the checksum's 8 uint16 lanes.
BANDS, LANES = 120, 8

def launch(cube_address, out_address, height, width):
    # Start the kernel on a height x width cube; every input port reads it.
    for offset, address in [(d, cube_address) for d in DINS] + [(DOUT, out_address)]:
        kernel.write(offset, address & 0xFFFFFFFF)
        kernel.write(offset + 4, address >> 32)
    kernel.write(HEIGHT, height)
    kernel.write(WIDTH, width)
    kernel.write(AP_CTRL, 1)

def wait(timeout=60):
    deadline = time.monotonic() + timeout
    while not kernel.read(AP_CTRL) & 0x2:      # done
        if time.monotonic() > deadline:
            raise TimeoutError('kernel did not finish')

MHZ = Clocks.fclk0_mhz                          # the kernel's clock, as the overlay set it
print(f'{NAME}, {len(DINS)} input ports, {MHZ:.1f} MHz, idle:', bool(kernel.read(AP_CTRL) & 0x4))

## 2. Load the image into DDR

The cube goes from the SD card into DDR the FPGA can read (PYNQ's CMA, shared by
every running notebook: shut down the ones you are not using). If the whole cube does
not fit, this notebook uses its first 128 rows.

In [ ]:
IMAGE = 'aegean_unet.zip'      # or an already unpacked folder
folder = Path(IMAGE)
if folder.suffix == '.zip':
    unpacked = Path(folder.stem) / 'cube.npy'
    if not unpacked.exists() or unpacked.stat().st_mtime < folder.stat().st_mtime:
        with zipfile.ZipFile(folder) as z:
            z.extractall('.')
    folder = Path(folder.stem)
t0 = time.monotonic()
cube = np.load(folder / 'cube.npy', mmap_mode='r')    # read from the SD card as it is used
if cube.dtype != np.uint16:                           # an image folder from before: counts as floats
    cube = np.rint(cube).astype(np.uint16)
STRIP = 128
try:
    cube_in = allocate(shape=cube.shape, dtype=np.uint16)
except Exception as error:                            # not enough (contiguous) CMA
    print(f'the whole cube does not fit in CMA ({error}): its first {STRIP} rows instead')
    cube = cube[:STRIP]
    cube_in = allocate(shape=cube.shape, dtype=np.uint16)
cube_in[:] = cube
cube_in.flush()                                       # into DDR, not just the CPU cache
out = allocate(shape=(LANES // 2,), dtype=np.int32)
H, W, _ = cube_in.shape
PATCHES = -(-H // 32) * -(-W // 32)
print(f'{H} x {W} pixels, {PATCHES} patches, loaded in {time.monotonic() - t0:.1f} s')

## 3. Check: every word arrived

The kernel adds up every word the four readers send, each of a word's 8 bands in its
own uint16 lane, mod 2^16, the edge patches' padding included (past the bottom the
last row again, past the right edge the last pixel). The same sum in numpy must match:
otherwise the readers did not read what `justounetsimple_opt` reads, and the time
below means nothing.

In [ ]:
def expected_checksum(c):
    h, w = c.shape[:2]
    down, right = -h % 32, -w % 32                    # padding rows and columns
    def lanes(a):
        return np.asarray(a).reshape(-1, LANES).sum(0, dtype=np.uint64)
    total = lanes(c) + down * lanes(c[-1]) + right * lanes(c[:, -1]) + down * right * lanes(c[-1, -1])
    return (total % 65536).astype(np.uint16)

out[:] = 0
out.flush()
launch(cube_in.physical_address, out.physical_address, H, W)
wait()
out.invalidate()
checksum = np.asarray(out).view(np.uint16)            # lane 2i in the low half of int32 i
expected = expected_checksum(cube_in)
ok = bool((checksum == expected).all())
print('PASS: every word arrived' if ok else f'FAIL: checksum {checksum}, expected {expected}')

## 4. Speed: the reads alone

The kernel again on the cube already in DDR, best of 5. **FPGA**: from the start bit
until the done bit, as in the `justounetsimple_opt` notebook's "FPGA alone", so the two
compare directly. The ideal: each port reads its 8 rows x 32 pixels x 15 words of a
patch, 3 840 cycles a patch at one word per cycle; `justounetsimple_opt`'s conv1 needs
4 129. The bandwidth counts the edge patches' padding too, as the kernel reads it.

In [ ]:
READ_CYCLES = 32 // len(DINS) * 32 * BANDS // 8    # a port's words of a patch, one per cycle
CONV1_CYCLES = 4 * 32 * 32 + 33                     # justounetsimple_opt's slowest compute stage
runs = []
for _ in range(5):
    t = time.perf_counter()
    launch(cube_in.physical_address, out.physical_address, H, W)
    wait()
    runs.append(time.perf_counter() - t)
best = min(runs)
cycles = best * MHZ * 1e6 / PATCHES
print(f'FPGA, reads only: {1000 * best:.2f} ms for {H} x {W} pixels ({PATCHES} patches), '
      f'{1e6 * best / PATCHES:.1f} us per patch')
print(f'  {cycles:.0f} cycles a patch at {MHZ:.0f} MHz; ideal {READ_CYCLES}, conv1 needs {CONV1_CYCLES}')
print(f'  {PATCHES * 32 * 32 * BANDS * 2 / best / 1e9:.2f} GB/s from DDR')